# 06 — QTc Methods

**Purpose:** Compute all four QTc corrections and quantify formula sensitivity.

**Output:** `outputs/qtc_comparison.parquet`

**Granularity:** Beat level — primary key: `record_id + beat_id`

**Data Contract:** `DATA_CONTRACT.md` §13

**No confidence calculations in this notebook.**


In [1]:
import sys
sys.path.insert(0, '../src')


In [2]:
import numpy as np
import pandas as pd
from datetime import datetime
from ecg_analytics.qtc.formulas import compute_all_qtc

RANDOM_SEED = 42
PIPELINE_VERSION = "1.0.0"
rng = np.random.default_rng(RANDOM_SEED)
TIMESTAMP = datetime.utcnow().isoformat()

qt_df = pd.read_parquet("../outputs/qt_measurements.parquet")
print(f"QT measurements loaded: {qt_df.shape}")
print(qt_df.head(3)[["record_id","beat_id","qt_ms","rr_ms"]].to_string(index=False))


QT measurements loaded: (7087, 10)
  record_id  beat_id  qt_ms  rr_ms
ptbxl/00001        0  232.0    NaN
ptbxl/00001        1  236.0  716.0
ptbxl/00001        2  266.0  690.0


/tmp/ipykernel_2791/577575463.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().isoformat()


## QTc Formula Comparison

Four standard formulas applied to every valid beat:

| Formula | Expression |
|---|---|
| Fridericia (default) | QT / RR^(1/3) |
| Bazett | QT / √RR |
| Framingham | QT + 154(1 − RR_s) |
| Hodges | QT + 1.75(HR − 60) |

`qtc_formula_variance` = variance across formulas for a single beat.
`qtc_formula_bias` = max − min spread (ms).


In [3]:
valid = qt_df.dropna(subset=["qt_ms","rr_ms"]).copy()
valid = valid[(valid["qt_ms"] > 200) & (valid["qt_ms"] < 700)]
valid = valid[(valid["rr_ms"]  > 300) & (valid["rr_ms"]  < 1800)]
print(f"Valid beats for QTc computation: {len(valid)}")

all_qtc = compute_all_qtc(valid["qt_ms"].values, valid["rr_ms"].values)

qtc_df = valid[["record_id","beat_id"]].copy()
qtc_df["rr_ms"]            = valid["rr_ms"].values
qtc_df["qtc_bazett"]       = all_qtc["bazett"]
qtc_df["qtc_fridericia"]   = all_qtc["fridericia"]
qtc_df["qtc_framingham"]   = all_qtc["framingham"]
qtc_df["qtc_hodges"]       = all_qtc["hodges"]

formula_cols = ["qtc_bazett","qtc_fridericia","qtc_framingham","qtc_hodges"]
qtc_values   = qtc_df[formula_cols].values
qtc_df["qtc_formula_variance"] = np.var(qtc_values, axis=1)
qtc_df["qtc_formula_bias"]     = qtc_values.max(axis=1) - qtc_values.min(axis=1)
qtc_df["pipeline_version"]     = PIPELINE_VERSION
qtc_df["processing_timestamp"] = TIMESTAMP

print(qtc_df[formula_cols + ["qtc_formula_bias"]].describe().round(2).to_string())


Valid beats for QTc computation: 6210
       qtc_bazett  qtc_fridericia  qtc_framingham  qtc_hodges  qtc_formula_bias
count     6210.00         6210.00         6210.00     6210.00           6210.00
mean       337.42          315.68          317.26      326.83             30.98
std        136.20          111.58           91.59      108.52             48.03
min        177.42          187.60          132.52      180.31              0.00
25%        263.84          255.48          266.02      263.66              8.60
50%        291.76          279.34          290.82      288.96             15.36
75%        339.55          314.05          324.55      334.08             24.11
max       1160.76          966.65          777.33      880.39            413.58


## Schema Validation

In [4]:
REQUIRED_QTC_COLS = [
    "record_id","beat_id","rr_ms",
    "qtc_bazett","qtc_fridericia","qtc_framingham","qtc_hodges",
    "qtc_formula_variance","qtc_formula_bias",
]
missing = [c for c in REQUIRED_QTC_COLS if c not in qtc_df.columns]
assert not missing, f"Missing cols: {missing}"
assert (qtc_df["qtc_formula_bias"] >= 0).all(), "Negative formula bias"
assert (qtc_df["qtc_formula_variance"] >= 0).all(), "Negative variance"
print("✓ Schema validation passed")


✓ Schema validation passed


## QTc Formula Visualisation

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {"qtc_fridericia":"#1f77b4","qtc_bazett":"#ff7f0e",
          "qtc_framingham":"#2ca02c","qtc_hodges":"#d62728"}

for col, color in colors.items():
    qtc_df[col].hist(bins=40, ax=axes[0], alpha=0.5, label=col.replace("qtc_","").title(),
                     color=color, edgecolor="none")
axes[0].set_title("QTc Distribution by Formula")
axes[0].legend(fontsize=7)

axes[1].scatter(qtc_df["rr_ms"], qtc_df["qtc_fridericia"], s=8, alpha=0.3, color="#1f77b4")
axes[1].set_xlabel("RR interval (ms)")
axes[1].set_ylabel("QTc Fridericia (ms)")
axes[1].set_title("QTc vs RR (Fridericia)")

qtc_df["qtc_formula_bias"].hist(bins=30, ax=axes[2], color="#9467bd", edgecolor="white")
axes[2].set_title("Formula Spread (max−min, ms)")
axes[2].set_xlabel("Bias (ms)")
plt.tight_layout()
plt.savefig("../outputs/qtc_comparison_summary.png", dpi=100)
plt.show()
print("Figure saved.")


Figure saved.


/tmp/ipykernel_2791/114191231.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export

In [6]:
qtc_df.to_parquet("../outputs/qtc_comparison.parquet", index=False)
print("✓ qtc_comparison.parquet →", qtc_df.shape)


✓ qtc_comparison.parquet → (6210, 11)
